# Predicting T-shirt size using the ANSUR II dataset
We will here try to predict a persons t-shirt size given the weight and height of the person. We will use the ANSUR II dataset which contains a lot of information about the physical attributes of a large number of people.
 
We will first try to map the persons in the dataset to a t-shirt size. It is hard to find a concise size chart for t-shirt so we will create our own, initial chart, based on these assumptions:
 
We will only look at two measurements, Shoulder Width and Chest Circumference.
 
Our first problem is that Shoulder Width is not one of the measurements taken in the dataset. But we have Biacromial Breadth which is the distance between the two acromion processes. We will assume that this is the same as Shoulder Width.
 
We will then have these initial rules:
 
| Size | Percentile |
|------|------------|
| XS   | 0-5        |
| S    | 5-25       |
| M    | 25-50      |
| L    | 50-75      |
| XL   | 75-90      |
| XXL  | 90-97      |
| XXXL | 97-100     |

# Lab 3

Earlier in the project, we mentioned that there might be conflicts when comparing sizes based on different measurements (e.g., chest circumference and shoulder breadth). For instance, a person might have size S for chest but size M for shoulders. Your task is to get a clearer picture of how many individuals have matching sizes for both measurements and how many have different sizes (i.e., they fall into different sizes for shoulder breadth and chest circumference).
 
Use the size chart: Use a size chart that specifies the limits for shoulder breadth and chest circumference for each size.
 
Create a function: Write a function that iterates through each person's measurements and compares them with the size chart.
 
Count matches and conflicts: The function should count the number of individuals who have exactly one matching size and the number of individuals who have multiple possible sizes (conflicts).
 
Test your function with both female and male datasets, and use appropriate size charts for each gender.

## Inspect the data

In [42]:
import pandas as pd
import numpy as np


In [43]:
female = pd.read_csv('./data/female.csv')
male = pd.read_csv('./data/male.csv')

In [44]:
print(f'For women we have (rows, columns) {female.shape}')
print(f'For men we have (rows, columns) {male.shape}')

For women we have (rows, columns) (1986, 108)
For men we have (rows, columns) (4082, 108)


## Checking the percentiles

In [45]:
def compute_percentile_ranges(column):
    # Define percentile ranges
    ranges = [(0,5), (5,25), (25,50), (50,75), (75,90), (90,97), (97,100)]
    
    percentiles = {(low, high): (column.quantile(low/100), column.quantile(high/100)) for low, high in ranges}
    
    counts = {}
    
    # range and values make up each items
    for r, (low, high) in percentiles.items():
        counts[r] = int(((column >= low) & (column < high)).sum())
        
    return counts
        
        
print(compute_percentile_ranges(female['chestcircumference']))
print(compute_percentile_ranges(female['biacromialbreadth']))

{(0, 5): 100, (5, 25): 396, (25, 50): 492, (50, 75): 499, (75, 90): 299, (90, 97): 140, (97, 100): 59}
{(0, 5): 93, (5, 25): 377, (25, 50): 477, (50, 75): 541, (75, 90): 297, (90, 97): 139, (97, 100): 61}


In [46]:

print(compute_percentile_ranges(male['chestcircumference']))
print(compute_percentile_ranges(male['biacromialbreadth']))

{(0, 5): 199, (5, 25): 810, (25, 50): 1025, (50, 75): 1012, (75, 90): 616, (90, 97): 295, (97, 100): 124}
{(0, 5): 191, (5, 25): 787, (25, 50): 989, (50, 75): 1079, (75, 90): 610, (90, 97): 303, (97, 100): 122}


## Generate the t-shirt size chart

In [47]:
def compute_size_percentile_measurements(data, chest_column, shoulder_column):
    sizes = ['XS', 'S', 'M', 'L', 'XL', '2XL', '3XL']
    ranges = [0, 5, 25, 50, 75, 90, 97]  # lowest percentile value for each
    
    # Compute the values for each percentile for chest and shoulder
    chest_percentiles = {p: data[chest_column].quantile(p/100) for p in ranges}
    shoulder_percentiles = {p: data[shoulder_column].quantile(p/100) for p in ranges}
    
    # Map the t-shirt sizes to the corresponding chest and shoulder measurements
    size_mappings = {}
    for i, size in enumerate(sizes):
        size_mappings[size] = {
            'Chest': int(chest_percentiles[ranges[i]]),
            'Shoulder': int(shoulder_percentiles[ranges[i]])
        }
    return size_mappings
    
print(compute_size_percentile_measurements(female, 'chestcircumference', 'biacromialbreadth'))
print(compute_size_percentile_measurements(male, 'chestcircumference', 'biacromialbreadth'))
    

{'XS': {'Chest': 695, 'Shoulder': 283}, 'S': {'Chest': 824, 'Shoulder': 335}, 'M': {'Chest': 889, 'Shoulder': 353}, 'L': {'Chest': 940, 'Shoulder': 365}, 'XL': {'Chest': 999, 'Shoulder': 378}, '2XL': {'Chest': 1057, 'Shoulder': 389}, '3XL': {'Chest': 1117, 'Shoulder': 400}}
{'XS': {'Chest': 774, 'Shoulder': 337}, 'S': {'Chest': 922, 'Shoulder': 384}, 'M': {'Chest': 996, 'Shoulder': 403}, 'L': {'Chest': 1056, 'Shoulder': 415}, 'XL': {'Chest': 1117, 'Shoulder': 428}, '2XL': {'Chest': 1172, 'Shoulder': 441}, '3XL': {'Chest': 1233, 'Shoulder': 452}}


In [48]:
female_sizes = {
    'XS': {'Chest': 695, 'Shoulder': 283}, 
    'S': {'Chest': 824, 'Shoulder': 335}, 
    'M': {'Chest': 889, 'Shoulder': 353}, 
    'L': {'Chest': 940, 'Shoulder': 365}, 
    'XL': {'Chest': 999, 'Shoulder': 378}, 
    '2XL': {'Chest': 1057, 'Shoulder': 389}, 
    '3XL': {'Chest': 1117, 'Shoulder': 400}
}
male_sizes = {
    'XS': {'Chest': 774, 'Shoulder': 337}, 
    'S': {'Chest': 922, 'Shoulder': 384}, 
    'M': {'Chest': 996, 'Shoulder': 403}, 
    'L': {'Chest': 1056, 'Shoulder': 415}, 
    'XL': {'Chest': 1117, 'Shoulder': 428}, 
    '2XL': {'Chest': 1172, 'Shoulder': 441}, 
    '3XL': {'Chest': 1233, 'Shoulder': 452}
}

# Lab 3 - Aladdins version

Earlier in the project, we mentioned that there might be conflicts when comparing sizes based on different measurements (e.g., chest circumference and shoulder breadth). For instance, a person might have size S for chest but size M for shoulders. Your task is to get a clearer picture of how many individuals have matching sizes for both measurements and how many have different sizes (i.e., they fall into different sizes for shoulder breadth and chest circumference).
 
Use the size chart: Use a size chart that specifies the limits for shoulder breadth and chest circumference for each size.
 
Create a function: Write a function that iterates through each person's measurements and compares them with the size chart.
 
Count matches and conflicts: The function should count the number of individuals who have exactly one matching size and the number of individuals who have multiple possible sizes (conflicts).
 
Test your function with both female and male datasets, and use appropriate size charts for each gender.

In [49]:
def get_size(data, size_chart):
    matches = {size: 0 for size in size_chart.keys()}
    ties = 0
    
    for _, row in data.iterrows():
        possible_sizes = []
        
        for size, measurements in size_chart.items():
            if (row['biacromialbreadth'] <= measurements['Shoulder'] and row['chestcircumference'] <= measurements['Chest']):
                possible_sizes.append(size)
        if len(possible_sizes) == 1:
            matches[possible_sizes[0]] += 1
        elif len(possible_sizes) > 1:
            ties += 1
    return matches, ties
            

female_matches, female_ties = get_size(female, female_sizes)

male_matches, male_ties = get_size(male, male_sizes)

print('Female matches:', female_matches)
print('Female ties:', female_ties)
print('Male matches:', male_matches)
print('Male ties:', male_ties)

Female matches: {'XS': 0, 'S': 0, 'M': 0, 'L': 0, 'XL': 0, '2XL': 0, '3XL': 236}
Female ties: 1642
Male matches: {'XS': 0, 'S': 0, 'M': 0, 'L': 0, 'XL': 0, '2XL': 0, '3XL': 434}
Male ties: 3437


# Lesson 4

Overlapping measurements

In [50]:
def create_overlapping_size_chart(original_chart):
    overlapping_chart = {}
    
    sizes = list(original_chart.keys())
    
    for i, size in enumerate(sizes):
        overlapping_chart[size] = {}
        if i == 0:
            overlapping_chart[size]['Chest'] = [original_chart[size]['Chest'], original_chart[sizes[i+1]]['Chest']+5]
            overlapping_chart[size]['Shoulder'] = [original_chart[size]['Shoulder'], original_chart[sizes[i+1]]['Shoulder']+5]
        elif i == len(sizes)-1:
            overlapping_chart[size]['Chest'] = [original_chart[size]['Chest']-5, original_chart[size]['Chest']+1000]
            overlapping_chart[size]['Shoulder'] = [original_chart[size]['Shoulder']-5, original_chart[size]['Shoulder']+1000]
        else:
            overlapping_chart[size]['Chest'] = [original_chart[size]['Chest']-5, original_chart[sizes[i+1]]['Chest']+5]
            overlapping_chart[size]['Shoulder'] = [original_chart[size]['Shoulder']-5, original_chart[sizes[i+1]]['Shoulder']+5]
            
    return overlapping_chart

new_female_sizes = create_overlapping_size_chart(female_sizes)
new_male_sizes = create_overlapping_size_chart(male_sizes)

print('new_female_sizes = {')
for k, v in new_female_sizes.items():
        print(f"'{k}': {v},")
print('}')
print('new_male_sizes = {')
for k, v in new_male_sizes.items():
        print(f"'{k}': {v},")
print('}')
    

new_female_sizes = {
'XS': {'Chest': [695, 829], 'Shoulder': [283, 340]},
'S': {'Chest': [819, 894], 'Shoulder': [330, 358]},
'M': {'Chest': [884, 945], 'Shoulder': [348, 370]},
'L': {'Chest': [935, 1004], 'Shoulder': [360, 383]},
'XL': {'Chest': [994, 1062], 'Shoulder': [373, 394]},
'2XL': {'Chest': [1052, 1122], 'Shoulder': [384, 405]},
'3XL': {'Chest': [1112, 2117], 'Shoulder': [395, 1400]},
}
new_male_sizes = {
'XS': {'Chest': [774, 927], 'Shoulder': [337, 389]},
'S': {'Chest': [917, 1001], 'Shoulder': [379, 408]},
'M': {'Chest': [991, 1061], 'Shoulder': [398, 420]},
'L': {'Chest': [1051, 1122], 'Shoulder': [410, 433]},
'XL': {'Chest': [1112, 1177], 'Shoulder': [423, 446]},
'2XL': {'Chest': [1167, 1238], 'Shoulder': [436, 457]},
'3XL': {'Chest': [1228, 2233], 'Shoulder': [447, 1452]},
}


Put the sizes into dictionaries

In [51]:
new_female_sizes = {
'XS': {'Chest': [695, 829], 'Shoulder': [283, 340]},
'S': {'Chest': [819, 894], 'Shoulder': [330, 358]},
'M': {'Chest': [884, 945], 'Shoulder': [348, 370]},
'L': {'Chest': [935, 1004], 'Shoulder': [360, 383]},
'XL': {'Chest': [994, 1062], 'Shoulder': [373, 394]},
'2XL': {'Chest': [1052, 1122], 'Shoulder': [384, 405]},
'3XL': {'Chest': [1112, 2117], 'Shoulder': [395, 1400]},
}
new_male_sizes = {
'XS': {'Chest': [774, 927], 'Shoulder': [337, 389]},
'S': {'Chest': [917, 1001], 'Shoulder': [379, 408]},
'M': {'Chest': [991, 1061], 'Shoulder': [398, 420]},
'L': {'Chest': [1051, 1122], 'Shoulder': [410, 433]},
'XL': {'Chest': [1112, 1177], 'Shoulder': [423, 446]},
'2XL': {'Chest': [1167, 1238], 'Shoulder': [436, 457]},
'3XL': {'Chest': [1228, 2233], 'Shoulder': [447, 1452]},
}

# Lab 4

Last time, we created a function get_size to get a clearer view of how many matches and ties we had. Now, I want you to do the same thing for the new size charts we created, but this time taking into consideration that we have two measurements instead of one. The goal remains the same: find out how many matches and how many ties we have.
 
Task
Analyze the data: Use the new size charts to determine the number of matches and ties based on two measurements.
Count matches and ties: Write a function that iterates through each person's measurements, compares them with the new size charts, and counts the number of matches and ties.
Bonus
Modify the function to handle ties. If there is a tie and the sizes are adjacent, choose the larger size to increase the number of matches.
 

In [52]:
def get_size(data, size_chart):
    matches = {size: 0 for size in size_chart.keys()}
    ties = 0
    
    size_ordered = list(size_chart.keys())
    print(size_ordered)
    
    for _, row in data.iterrows():
        possible_sizes = []
        
        for size, measurements in size_chart.items():
            if (row['biacromialbreadth'] >= measurements['Shoulder'][0] and 
                row['biacromialbreadth'] <= measurements['Shoulder'][1] and
                row['chestcircumference'] >= measurements['Chest'][0] and
                row['chestcircumference'] <= measurements['Chest'][1]):
                possible_sizes.append(size)
                
        if len(possible_sizes) == 1:
            matches[possible_sizes[0]] += 1
        elif len(possible_sizes) > 1:
            # Check if sizes are adjacent
            are_adjacent = all([abs(size_ordered.index(possible_sizes[i]) - size_ordered.index(possible_sizes[i+1])) == 1 for i in range(len(possible_sizes) -1 )])
           
            if are_adjacent:
                #assign the larger size
                larger_size = max(possible_sizes, key=lambda s: size_ordered.index(s))
                matches[larger_size] += 1
            else:
                ties += 1
                
    return matches, ties

In [53]:
# example of all function
result = all([True, True, False])
print(result)

x = 10
result = all([x < 15, x == 10, x > 12])
print(result)



False
False


In [54]:

female_matches, female_ties = get_size(female, new_female_sizes)

male_matches, male_ties = get_size(male, new_male_sizes)

print('Female matches:', female_matches)
print('Female ties:', female_ties)
print('Male matches:', male_matches)
print('Male ties:', male_ties)

['XS', 'S', 'M', 'L', 'XL', '2XL', '3XL']
['XS', 'S', 'M', 'L', 'XL', '2XL', '3XL']
Female matches: {'XS': 23, 'S': 185, 'M': 247, 'L': 276, 'XL': 118, '2XL': 35, '3XL': 13}
Female ties: 0
Male matches: {'XS': 63, 'S': 428, 'M': 578, 'L': 593, 'XL': 331, '2XL': 101, '3XL': 50}
Male ties: 0


# Lesson 5


In [55]:
def determine_size(value, measurement, size_dict):
    """
    Determine clothing sizes that match the given body measurement value.

    Iterates through the provided size dictionary and finds all sizes where the 
    measurement range contains the given value.

    Parameters:
      value: The body measurement value
      measurement: The name of the measurement type
      size_dict: Dictionary mapping clothing sizes to measurement ranges

    Returns:
      sizes: List of clothing sizes matching the measurement value
    """
    sizes = []
    for size, measurements in size_dict.items():
        if measurements[measurement][0] <= value <= measurements[measurement][1]:
            sizes.append(size)
    return sizes

def determine_individual_size(row, size_dict):
    """
    Determine individual clothing size based on chest circumference and shoulder width.
    Matches the chest and shoulder measurements to clothing sizes in the provided size dictionary. 
    Returns a single matching size, or the larger of two adjacent sizes if there are multiple matches.
    Returns None if no size matches or adjacent sizes cannot be determined.
    """
    chest_size = determine_size(row['chestcircumference'], 'Chest', size_dict)
    shoulder_size = determine_size(row['biacromialbreadth'], 'Shoulder', size_dict)

    matching_sizes = list(set(chest_size) & set(shoulder_size))

    if len(matching_sizes) == 1:
        return matching_sizes[0]
    elif len(matching_sizes) > 1:
        # Check if sizes are adjacent, if so, select the larger size
        size_order = list(size_dict.keys())
        adjacent = all([abs(size_order.index(a) - size_order.index(b)) <= 1 for a in matching_sizes for b in matching_sizes])
        if adjacent:
            return max(matching_sizes, key=lambda x: size_order.index(x))
    return None

In [56]:
female['t-shirtsize'] = female.apply(determine_individual_size, args=(new_female_sizes,), axis=1)
male['t-shirtsize'] = male.apply(determine_individual_size, args=(new_male_sizes,), axis=1)

In [57]:
female['t-shirtsize'].value_counts()

t-shirtsize
L      276
M      247
S      185
XL     118
2XL     35
XS      23
3XL     13
Name: count, dtype: int64

In [58]:

male['t-shirtsize'].value_counts()

t-shirtsize
L      593
M      578
S      428
XL     331
2XL    101
XS      63
3XL     50
Name: count, dtype: int64

In [59]:
# dropna to remove missing values
female_filtered = female.dropna(subset=['t-shirtsize'])
male_filtered = male.dropna(subset=['t-shirtsize'])

In [60]:
female_filtered.shape

(897, 109)

In [61]:
male_filtered.shape

(2144, 109)

In [65]:
columns = ['chestcircumference', 'biacromialbreadth', 'weightkg', 'stature', 't-shirtsize']

female_dataset = female_filtered[columns]
male_dataset = male_filtered[columns]

female_dataset.to_csv('./data/female_sized.csv', index=False)
male_dataset.to_csv('./data/male_sized.csv', index=False)

### Why create a Scatter Plot?
* Visualize the Distribution of T-shirt Sizes 
* Identify Patterns or Trends
* Compare Men and Women
* Communicate Results
* Identify Potential Anomalies or Outliers

In [ ]:
# import matplotlib.pyplot as plt


# colors = {
#     'XS': 'red',
#     'S': 'blue',
#     'M': 'green',
#     'L': 'yellow',
#     'XL': 'purple',
#     '2XL': 'cyan',
#     '3XL': 'magenta',
# }

# female_dataset = pd.read_csv('./data/female_sized.csv')
# male_dataset = pd.read_csv('./data/male_sized.csv')

# female_dataset.loc[:, 'stature'] = female_dataset['stature'] / 10
# female_dataset.loc[:, 'weightkg'] = female_dataset['weightkg'] / 10

# male_dataset.loc[:, 'stature'] = male_dataset['stature'] / 10
# male_dataset.loc[:, 'weightkg'] = male_dataset['weightkg'] / 10

# fig, axes = plt.subplots(nrow=2, figsize=(10,12))

# for ax, gender_data, gender in zip(axes, [male_dataset, female_dataset], {'Male:}):
    

In [ ]:
# class KNNClassifier: 
    